# HyperMiner notebook (`nb-hyperminer-ppd.ipynb`)

Notebook compatible **Kaggle + PC local**.

Objectif:
1. utiliser les 4 datasets (`20NG`, `AGNews`, `IMDB`, `YahooAnswer`),
2. entra?ner HyperMiner pour `K in [20, 50, 100]`,
3. exporter des fichiers `.mat` harmonis?s avec:
- `beta`
- `train_theta`
- `test_theta`

Note:
- on utilise un setup 1-couche (`num_topics_list=[K]`) pour harmoniser directement les sorties avec les autres mod?les.


In [ ]:
import os
import sys
import random
from pathlib import Path
from types import SimpleNamespace

import numpy as np
import scipy
import scipy.io
import scipy.sparse as sp
import torch
from torch.utils.data import Dataset, DataLoader

print("Python:", sys.version.split()[0])
print("NumPy:", np.__version__)
print("SciPy:", scipy.__version__)
print("Torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

In [ ]:

import os
from pathlib import Path

TARGET_DATASETS = ["20NG", "AGNews", "IMDB", "YahooAnswer"]
K_LIST = [20, 50, 100]
RANDOM_SEED = 42

REQUIRED_DATA_FILES = [
    "train_bow.npz",
    "test_bow.npz",
    "train_labels.txt",
    "test_labels.txt",
    "vocab.txt",
    "word_embeddings.npz",
]

IS_KAGGLE = Path("/kaggle").exists()


def find_project_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "ECTRM_source").exists():
            return candidate
    return start


if IS_KAGGLE:
    PROJECT_ROOT = Path("/kaggle/working")
    INPUT_ROOT = Path("/kaggle/input")
else:
    PROJECT_ROOT = find_project_root(Path.cwd())
    INPUT_ROOT = PROJECT_ROOT

print("IS_KAGGLE:", IS_KAGGLE)
print("PROJECT_ROOT:", PROJECT_ROOT)
print("INPUT_ROOT:", INPUT_ROOT)


def has_required_files(path: Path) -> bool:
    return path.exists() and path.is_dir() and all((path / f).exists() for f in REQUIRED_DATA_FILES)


def guess_dataset_name(path: Path):
    s = str(path).lower()
    if "20ng" in s or "20news" in s:
        return "20NG"
    if "agnews" in s or "ag_news" in s:
        return "AGNews"
    if "yahoo" in s:
        return "YahooAnswer"
    if "imdb" in s:
        return "IMDB"
    return None


def iter_candidate_dirs(base: Path, max_depth: int = 5):
    if not base.exists():
        return
    base_depth = len(base.parts)
    for root, dirs, _ in os.walk(base):
        root_path = Path(root)
        depth = len(root_path.parts) - base_depth
        if depth > max_depth:
            dirs[:] = []
            continue
        yield root_path


def discover_dataset_dirs():
    found = {}

    # Local canonical layout
    local_data_root = PROJECT_ROOT / "ECTRM_source" / "ECRTM" / "data"
    for ds in TARGET_DATASETS:
        p = local_data_root / ds
        if has_required_files(p):
            found[ds] = p

    # Kaggle / custom layout scan
    scan_roots = [Path("/kaggle/input"), PROJECT_ROOT, INPUT_ROOT] if IS_KAGGLE else [PROJECT_ROOT, INPUT_ROOT]
    seen = set()
    for scan_root in scan_roots:
        for cand in iter_candidate_dirs(scan_root):
            cand_s = str(cand)
            if cand_s in seen:
                continue
            seen.add(cand_s)
            if not has_required_files(cand):
                continue
            ds = guess_dataset_name(cand)
            if ds is not None and ds not in found:
                found[ds] = cand

    return found


DATASET_DIRS = discover_dataset_dirs()
print("\nResolved dataset directories:")
for ds in TARGET_DATASETS:
    print(f" - {ds}:", DATASET_DIRS.get(ds))

missing = [ds for ds in TARGET_DATASETS if ds not in DATASET_DIRS]
if missing:
    print("\nWARNING missing datasets:", missing)


In [ ]:

MODEL_NAME = "HyperMiner"
OUTPUT_OS_DIR = PROJECT_ROOT / "Sortie_mat" / MODEL_NAME
OUTPUT_KAGGLE_DIR = Path("/kaggle/working/Sortie_mat/output/kaggle") / MODEL_NAME

OUTPUT_DIR = OUTPUT_KAGGLE_DIR if IS_KAGGLE else OUTPUT_OS_DIR
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("OUTPUT_DIR:", OUTPUT_DIR)


In [ ]:
HYPERMINER_DEFAULTS = {
    "epochs": 300,
    "batch_size": 200,
    "lr": 1e-2,
    "weight_decay": 1e-5,
    "hidden_size": 300,
    "act": "relu",
    "manifold": "PoincareBall",  # Euclidean / Hyperboloid / PoincareBall
    "c": -0.01,
    "clip_r": None,
    "grad_clip": 20.0,
}

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("DEVICE:", DEVICE)
print("HYPERMINER_DEFAULTS:", HYPERMINER_DEFAULTS)

In [ ]:
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def load_vocab(path: Path):
    with open(path, "r", encoding="utf-8") as f:
        return [line.strip() for line in f if line.strip()]


def load_bow_csr(path: Path):
    return sp.load_npz(path).astype(np.float32).tocsr()


def load_word_embeddings(path: Path):
    return sp.load_npz(path).astype(np.float32).toarray()


class BowDataset(Dataset):
    def __init__(self, bow_csr, labels):
        self.bow = bow_csr.tocsr()
        self.labels = labels.astype(np.int32)

    def __len__(self):
        return self.bow.shape[0]

    def __getitem__(self, idx):
        row = self.bow[idx].toarray().astype(np.float32).squeeze(0)
        return torch.from_numpy(row), int(self.labels[idx])

In [ ]:
hyper_repo = PROJECT_ROOT / "HyperMiner_source"
if not hyper_repo.exists():
    raise FileNotFoundError(f"HyperMiner source introuvable: {hyper_repo}")

sys.path.insert(0, str(hyper_repo))

from models.hyperminer import HyperMiner

print("HyperMiner repo:", hyper_repo)

In [ ]:
def build_hyperminer_model(K, vocab_size, word_emb, cfg):
    embed_size = int(word_emb.shape[1])

    args = SimpleNamespace(
        embed_size=embed_size,
        vocab_size=vocab_size,
        num_topics_list=[int(K)],
        num_hiddens_list=[int(cfg["hidden_size"])],
        act=cfg["act"],
        manifold=cfg["manifold"],
        c=cfg["c"],
        clip_r=cfg["clip_r"],
    )

    model = HyperMiner(args, DEVICE, word_emb)
    return model.to(DEVICE)


def train_hyperminer(model, train_loader, cfg, seed=42):
    set_seed(seed)

    optimizer = torch.optim.Adam(
        params=model.parameters(),
        lr=float(cfg["lr"]),
        weight_decay=float(cfg["weight_decay"]),
    )

    for epoch in range(int(cfg["epochs"])):
        model.train()
        losses = []

        for bow, _ in train_loader:
            bow = bow.to(DEVICE).float()

            nelbo, _, _, _ = model(bow, is_training=True)
            optimizer.zero_grad()
            nelbo.backward()

            if cfg.get("grad_clip") is not None:
                torch.nn.utils.clip_grad_norm_(model.parameters(), float(cfg["grad_clip"]))

            optimizer.step()
            losses.append(float(nelbo.item()))

        if (epoch + 1) % 10 == 0 or epoch == 0:
            print(f"Epoch {epoch+1:03d}/{cfg['epochs']} - nelbo={np.mean(losses):.4f}")


def infer_theta_loader(model, loader):
    model.eval()
    all_theta = []

    with torch.no_grad():
        for bow, _ in loader:
            bow = bow.to(DEVICE).float()
            _, _, _, thetas = model(bow, is_training=False)
            theta = thetas[0]
            theta = theta / torch.clamp(theta.sum(1, keepdim=True), min=1e-8)
            all_theta.append(theta.cpu().numpy().astype(np.float32))

    return np.vstack(all_theta)


def run_one_hyperminer(dataset, K, seed=42, cfg=None):
    cfg = dict(HYPERMINER_DEFAULTS if cfg is None else cfg)

    if dataset not in DATASET_DIRS:
        raise KeyError(f"Dataset {dataset} introuvable")

    data_dir = DATASET_DIRS[dataset]

    train_csr = load_bow_csr(data_dir / "train_bow.npz")
    test_csr = load_bow_csr(data_dir / "test_bow.npz")
    train_labels = np.loadtxt(data_dir / "train_labels.txt", dtype=np.int32)
    test_labels = np.loadtxt(data_dir / "test_labels.txt", dtype=np.int32)
    word_emb = load_word_embeddings(data_dir / "word_embeddings.npz")

    train_ds = BowDataset(train_csr, train_labels)
    test_ds = BowDataset(test_csr, test_labels)

    train_loader = DataLoader(train_ds, batch_size=int(cfg["batch_size"]), shuffle=True, num_workers=0)
    test_loader = DataLoader(test_ds, batch_size=int(cfg["batch_size"]), shuffle=False, num_workers=0)

    model = build_hyperminer_model(K, train_csr.shape[1], word_emb, cfg)

    print(f"\n=== HyperMiner {dataset} K={K} ===")
    train_hyperminer(model, train_loader, cfg, seed=seed)

    with torch.no_grad():
        phis = model.get_phi()
        if isinstance(phis, list):
            phi0 = phis[0].detach().cpu().numpy().astype(np.float32)  # V x K
        else:
            phi0 = phis.detach().cpu().numpy().astype(np.float32)

    beta = phi0.T
    train_theta = infer_theta_loader(model, train_loader)
    test_theta = infer_theta_loader(model, test_loader)

    ds_out = OUTPUT_DIR / dataset
    ds_out.mkdir(parents=True, exist_ok=True)
    out_path = ds_out / f"{dataset}_HyperMiner_K{K}_seed{seed}.mat"

    scipy.io.savemat(
        str(out_path),
        {
            "beta": beta,
            "train_theta": train_theta,
            "test_theta": test_theta,
            "traintheta": train_theta,
            "testtheta": test_theta,
        }
    )
    print("Saved:", out_path)
    return out_path

In [ ]:
for dataset in TARGET_DATASETS:
    if dataset not in DATASET_DIRS:
        print("SKIP missing dataset:", dataset)
        continue
    for K in K_LIST:
        run_one_hyperminer(dataset, K=K, seed=RANDOM_SEED)

In [ ]:
import pandas as pd
from sklearn import metrics


def purity_score(y_true, y_pred):
    contingency_matrix = metrics.cluster.contingency_matrix(y_true, y_pred)
    return np.sum(np.amax(contingency_matrix, axis=0)) / np.sum(contingency_matrix)


def topic_diversity_from_beta(beta, topn=25):
    all_words = []
    for k in range(beta.shape[0]):
        top_idx = np.argsort(beta[k])[::-1][:topn]
        all_words.extend(top_idx.tolist())
    return len(set(all_words)) / max(1, len(all_words))


rows = []
for dataset in TARGET_DATASETS:
    if dataset not in DATASET_DIRS:
        continue

    labels = np.loadtxt(DATASET_DIRS[dataset] / "test_labels.txt", dtype=np.int32)
    ds_out = OUTPUT_DIR / dataset

    for K in K_LIST:
        mat_path = ds_out / f"{dataset}_HyperMiner_K{K}_seed{RANDOM_SEED}.mat"
        if not mat_path.exists():
            continue

        loaded = scipy.io.loadmat(str(mat_path))
        beta = loaded["beta"]
        test_theta = loaded["test_theta"]

        n_eval = min(len(labels), test_theta.shape[0])
        y_true = labels[:n_eval]
        y_pred = np.argmax(test_theta[:n_eval], axis=1)

        rows.append({
            "model": "HyperMiner",
            "dataset": dataset,
            "K": K,
            "Purity": float(purity_score(y_true, y_pred)),
            "NMI": float(metrics.cluster.normalized_mutual_info_score(y_true, y_pred)),
            "TD_top25": float(topic_diversity_from_beta(beta, topn=25)),
        })


if rows:
    df = pd.DataFrame(rows).sort_values(["dataset", "K"]).reset_index(drop=True)
    display(df)
    csv_path = OUTPUT_DIR / "HyperMiner_metrics_TD_Purity_NMI.csv"
    df.to_csv(csv_path, index=False)
    print("Saved:", csv_path)
else:
    print("No HyperMiner .mat results found yet.")

In [ ]:
TOPN = 15
for dataset in TARGET_DATASETS:
    ds_out = OUTPUT_DIR / dataset
    if not ds_out.exists() or dataset not in DATASET_DIRS:
        continue

    vocab = load_vocab(DATASET_DIRS[dataset] / "vocab.txt")
    for K in K_LIST:
        mat_path = ds_out / f"{dataset}_HyperMiner_K{K}_seed{RANDOM_SEED}.mat"
        if not mat_path.exists():
            continue
        loaded = scipy.io.loadmat(str(mat_path))
        beta = loaded["beta"]

        txt_path = ds_out / f"topics_for_cv_{dataset}_HyperMiner_K{K}_seed{RANDOM_SEED}.txt"
        with open(txt_path, "w", encoding="utf-8") as f:
            for k in range(beta.shape[0]):
                top_idx = np.argsort(beta[k])[::-1][:TOPN]
                f.write(" ".join(vocab[i] for i in top_idx) + "\n")
        print("Saved:", txt_path)

print("\nFiles in OUTPUT_DIR:")
for p in sorted(OUTPUT_DIR.rglob("*")):
    if p.is_file():
        print(" -", p.relative_to(OUTPUT_DIR))